STEDI: Curate Labeled Step Dataset (Device Messages + Rapid Step Tests)

Goal: merge device sensor readings with step-test sessions, then label each sensor row as step or no_step using timestamps, and keep clean source labels.

In [0]:
%python
### Confirm where you are (catalog/schema) and what tables actually exist ###

# Where am I?
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

# What’s in the bronze schema of the workspace catalog?
spark.sql("SHOW TABLES IN workspace.bronze").show(200, truncate=False)


In [0]:
### Load the two tables (using the names your Catalog actually has) ###
df_device = spark.table("workspace.bronze.device_messages_raw")
df_steps  = spark.table("workspace.bronze.rapid_step_tests_raw")

display(df_device)
display(df_steps)

Notes about expected columns

The doc describes: timestamp, sensorType, distance ("1cm"), deviceId, plus startTime/stopTime in the step table. In practice, your tables may use either camelCase or snake_case. Next cell normalizes column names so the joins don’t silently fail.

In [0]:
### Normalize column names and types (defensive, saves you pain) ###
from pyspark.sql.functions import col, to_timestamp

def rename_if_exists(df, old, new):
    return df.withColumnRenamed(old, new) if old in df.columns else df

# Device table normalization
df_device_n = df_device
df_device_n = rename_if_exists(df_device_n, "deviceId", "device_id")
df_device_n = rename_if_exists(df_device_n, "sensorType", "sensor_type")
# timestamp sometimes comes as string; force to timestamp when possible
df_device_n = rename_if_exists(df_device_n, "timestamp", "timestamp")
df_device_n = df_device_n.withColumn("timestamp", to_timestamp(col("timestamp")))

# Steps table normalization
df_steps_n = df_steps
df_steps_n = rename_if_exists(df_steps_n, "deviceId", "device_id")
df_steps_n = rename_if_exists(df_steps_n, "startTime", "start_time")
df_steps_n = rename_if_exists(df_steps_n, "stopTime", "stop_time")
df_steps_n = df_steps_n.withColumn("start_time", to_timestamp(col("start_time")))
df_steps_n = df_steps_n.withColumn("stop_time",  to_timestamp(col("stop_time")))

display(df_device_n)
display(df_steps_n)


In [0]:
### Clean / convert distance: "1cm" -> distance_cm (int) ###
from pyspark.sql.functions import regexp_extract

df_device_clean = df_device_n.withColumn(
    "distance_cm",
    regexp_extract(col("distance"), r"(\d+)", 1).cast("int")
)

display(df_device_clean.select("distance", "distance_cm").limit(20))


In [0]:
### Add source labels (so every row is traceable) ###
from pyspark.sql.functions import lit

df_device_labeled_source = df_device_clean.withColumn("source", lit("device"))
df_steps_labeled_source  = df_steps_n.withColumn("source", lit("step"))


In [0]:
# Cell 8 — Build step windows with a non-colliding device id name (ran into a bug early on for a future step where the device id name was two separate instances, which confused databricks)
df_steps_window = (
    df_steps_labeled_source
    .select("device_id", "start_time", "stop_time")
    .dropna(subset=["device_id", "start_time", "stop_time"])
    .withColumnRenamed("device_id", "step_device_id")
)

display(df_steps_window)


In [0]:
### Label each device sensor reading as step or no_step using timestamp BETWEEN start_time and stop_time ###
from pyspark.sql.functions import when, lit, col

df_labeled = (
    df_device_labeled_source.alias("d")
    .join(
        df_steps_window.alias("s"),
        (col("d.device_id") == col("s.step_device_id")) &
        (col("d.timestamp").between(col("s.start_time"), col("s.stop_time"))),
        "left"
    )
    .withColumn(
        "step_label",
        when(col("s.start_time").isNotNull(), lit("step")).otherwise(lit("no_step"))
    )
)

display(df_labeled)



Sanity check: did the join actually match anything?

If the step_label is 100% no_step, it usually means device_id values don’t match (string/int mismatch) or timestamps are in different formats/timezones. Next cell gives us the quick debug views.

In [0]:
### Quick debugging helpers (use only if your labels look wrong) ###
from pyspark.sql.functions import col

# Count label distribution
df_labeled.groupBy("step_label").count().show()

# Peek at a few rows where it labeled "step"
display(
    df_labeled
    .filter(col("step_label")=="step")
    .select("device_id", "timestamp", "step_label")
    .limit(50)
)

# Peek at step windows for the same device_id values
step_device_ids = [r["step_device_id"] for r in df_steps_window.select("step_device_id").distinct().limit(5).collect()]
display(
    df_steps_window
    .filter(col("step_device_id").isin(step_device_ids))
    .orderBy("step_device_id", "start_time")
    .limit(100)
)



In [0]:
### Select only the required final columns ###
df_final = df_labeled.select(
    "timestamp",
    "sensor_type",
    "distance_cm",
    "device_id",
    "step_label",
    "source"
)

display(df_final)


In [0]:
### Check the timestamp; this is ENTIRELY a debugging step and should only be required if the timestamp is trying to be a string rather than a Spark timestamp. ###
from pyspark.sql.functions import col, to_timestamp

df_final = df_final.withColumn("timestamp", to_timestamp(col("timestamp")))


In [0]:
%sql
--Drop table first; since the data is frozen, this should be safe to do and will resolve some errors with trying to create this table.
DROP TABLE IF EXISTS workspace.silver.labeled_step_test;


In [0]:
### Save the curated dataset as a table ###
target_table = "workspace.silver.labeled_step_test"

df_final.write.mode("overwrite").saveAsTable(target_table)

print("Saved to:", target_table)


# Verification

Need to ensure that these checks return no rows of missing or corrupted data. If they do return incorrect data, then we need to review the cells above to ensure the filters are functioning as expected.

In [0]:
%sql
--Count instances of step vs no_step
SELECT
  step_label,
  COUNT(*) AS row_count
FROM workspace.silver.labeled_step_test
GROUP BY step_label;


In [0]:
%sql
--Hunt for bad or missing step labels (should return 0 rows)
SELECT *
FROM workspace.silver.labeled_step_test
WHERE step_label NOT IN ('step', 'no_step')
   OR step_label IS NULL
LIMIT 50;


In [0]:
%sql
--Count the source labels
SELECT
  source,
  COUNT(*) AS row_count
FROM workspace.silver.labeled_step_test
GROUP BY source;


In [0]:
%sql
-- hunt for bad or missing source labels (should return 0 rows)
SELECT *
FROM workspace.silver.labeled_step_test
WHERE source NOT IN ('device', 'step')
   OR source IS NULL
LIMIT 50;


# Ethics Statement
I labeled step vs no_step based strictly on observed timestamps from a recorded step-test session, and I kept the labeling logic reproducible and visible in code. I treated device_id as a technical identifier and avoided adding any extra identifying fields beyond what the dataset already contained. I described the output as a labeled dataset for activity detection and avoided making any health or medical claims about a user. According to the actions taken, I believe we are:
1) Labeling data fairly
2) Protecting the identity of anyone inside the data
3) Avoiding medical claims